# AI & Data Jobs Market: Salaries, Skills, and Demand Trends

> **36K job listings, 280 companies, 318K job-skill edges** | [Dataset](https://www.kaggle.com/datasets/lorenzoscaturchio/ai-data-jobs-skills-salaries-2024-2026)

**Problem.** Compensation for AI and data roles is opaque: salary varies wildly with seniority, geography, company stage, and the exact mix of skills a posting demands. This notebook turns five linked tables into a coherent picture of *what pays*, *what is in demand*, and *how the market is shifting* between 2024 and 2026.

**Data.** Five relational tables: `jobs.csv` (one row per listing), `companies.csv` (one row per employer), `job_skills.csv` (a ~318K-edge job-to-skill graph), `salary_benchmarks.csv` (pre-aggregated percentile bands), and `skill_demand_monthly.csv` (month-by-skill demand).

**Approach.** We profile and clean the three core tables, run targeted EDA (salary distribution, salary by seniority, top in-demand skills, skill pay premiums), then fit a cross-validated **salary regression** with a transparent feature set, rank skills via the edge table, and study demand trends. We close with interpreted insights, caveats, and next steps.

## Table of Contents
1. [Objective](#objective)
2. [Setup & Loading](#setup)
3. [Reproducibility](#repro)
4. [Dataset Inventory](#inventory)
5. [Data Quality Check](#quality)
6. [Role and Salary Landscape](#roles)
7. [Salary Distribution (EDA)](#dist)
8. [Salary by Seniority (EDA)](#seniority)
9. [Skill Demand and Trend Shifts](#skills)
10. [Skill Ranking via the Edge Table](#ranking)
11. [Skill Pay Premiums](#premium)
12. [Benchmarks and Remote Mix](#benchmarks)
13. [Method: Salary Regression](#method)
14. [Evaluation](#evaluation)
15. [Diagnostics: Residuals & Feature Importance](#diagnostics)
16. [Insights](#insights)
17. [Conclusion & Next Steps](#conclusion)

---

## 1. Objective <a id='objective'></a>

Our goal is to make the linked jobs-market tables immediately useful for three concrete tasks:

- **Salary regression** &mdash; predict `salary_mid_usd` from role, seniority, geography, company attributes, and skill breadth, with honest cross-validated error bars.
- **Skill-demand analysis** &mdash; rank skills by how often listings require them (the edge table) and track how demand shifts month over month from 2024 to 2026.
- **Company and market ranking** &mdash; surface where pay concentrates by role family, country, and remote arrangement.

We favour interpretable, reproducible analysis over leaderboard-chasing: every chart and model is meant to support a defensible statement about the AI/data labour market.

In [ ]:
TARGET_COL = 'salary_mid_usd'
MODELING_TASK = 'multi-table exploration with regression, trend analysis, and ranking use cases'
PRIMARY_METRIC = 'RMSE / MAE for salary regression, edge-count ranking for skills, descriptive trend checks for demand'
VALIDATION_PLAN = 'K-fold cross-validation for the salary model; grouped joins for skill edges'

print('Explorer framing')
print('-' * 70)
print(f'Target candidate   : {TARGET_COL}')
print(f'Primary task       : {MODELING_TASK}')
print(f'Validation default : {VALIDATION_PLAN}')
print(f'Primary metric     : {PRIMARY_METRIC}')

## 2. Setup & Loading <a id='setup'></a>

We load the five tables from `/kaggle/input` when running on Kaggle, falling back to a `kagglehub` download elsewhere. Plot styling is set once so every chart below is consistent and legible.

In [ ]:
import subprocess
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
sns.set_palette('deep')

candidate_dirs = [
    Path('/kaggle/input/ai-data-jobs-skills-salaries-2024-2026'),
    Path('/kaggle/input/ai-data-jobs-market'),
    Path('.'),
]

input_root = Path('/kaggle/input')
if input_root.exists():
    discovered = [p for p in input_root.iterdir() if p.is_dir() and (p / 'jobs.csv').exists() and (p / 'companies.csv').exists()]
    candidate_dirs = discovered + candidate_dirs

DATA_DIR = next((p for p in candidate_dirs if (p / 'jobs.csv').exists()), None)
DOWNLOAD_DIR = None

if DATA_DIR is None:
    try:
        import kagglehub
    except ImportError:  # pragma: no cover - Kaggle runtime fallback
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
        import kagglehub

    downloaded = Path(kagglehub.dataset_download('lorenzoscaturchio/ai-data-jobs-skills-salaries-2024-2026'))
    discovered_downloads = [downloaded] + list(downloaded.rglob('*'))
    DATA_DIR = next((p for p in discovered_downloads if p.is_dir() and (p / 'jobs.csv').exists() and (p / 'companies.csv').exists()), None)
    DOWNLOAD_DIR = downloaded

if DATA_DIR is None:
    raise FileNotFoundError('Could not locate jobs-market CSV files in /kaggle/input or via kagglehub.dataset_download.')

jobs = pd.read_csv(DATA_DIR / 'jobs.csv')
companies = pd.read_csv(DATA_DIR / 'companies.csv')
job_skills = pd.read_csv(DATA_DIR / 'job_skills.csv')
salary_benchmarks = pd.read_csv(DATA_DIR / 'salary_benchmarks.csv')
skill_demand_monthly = pd.read_csv(DATA_DIR / 'skill_demand_monthly.csv')

print(f'Data directory        : {DATA_DIR}')
if DOWNLOAD_DIR is not None:
    print(f'kagglehub cache root : {DOWNLOAD_DIR}')
print(f'jobs                 : {jobs.shape[0]:,} rows x {jobs.shape[1]} columns')
print(f'companies            : {companies.shape[0]:,} rows x {companies.shape[1]} columns')
print(f'job_skills           : {job_skills.shape[0]:,} rows x {job_skills.shape[1]} columns')
print(f'salary_benchmarks    : {salary_benchmarks.shape[0]:,} rows x {salary_benchmarks.shape[1]} columns')
print(f'skill_demand_monthly : {skill_demand_monthly.shape[0]:,} rows x {skill_demand_monthly.shape[1]} columns')

## 3. Reproducibility <a id='repro'></a>

Every random operation in this notebook &mdash; train/test splits, K-fold shuffling, and model initialisation &mdash; is pinned to a single `SEED`. Fixing the seed once and threading it through `random_state` arguments means the salary model, its cross-validation folds, and any sampled diagnostics reproduce exactly on re-run.

In [ ]:
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# A single rng we can pass anywhere a NumPy Generator is accepted.
rng = np.random.default_rng(SEED)

print(f'Global SEED set to {SEED} (random, numpy, default_rng all pinned).')
print('All model fits and CV folds below use random_state=SEED for exact reproducibility.')

## 4. Dataset Inventory <a id='inventory'></a>

Before any modelling we confirm the grain of each table and eyeball the core columns. The `jobs` table is the analytical centre; the other four enrich it through `job_id`, `company_id`, and `skill` keys.

In [ ]:
file_inventory = pd.DataFrame([
    {'table': 'jobs', 'rows': len(jobs), 'columns': jobs.shape[1], 'grain': 'one row per job listing'},
    {'table': 'companies', 'rows': len(companies), 'columns': companies.shape[1], 'grain': 'one row per company'},
    {'table': 'job_skills', 'rows': len(job_skills), 'columns': job_skills.shape[1], 'grain': 'one row per job-skill edge'},
    {'table': 'salary_benchmarks', 'rows': len(salary_benchmarks), 'columns': salary_benchmarks.shape[1], 'grain': 'grouped benchmark slice'},
    {'table': 'skill_demand_monthly', 'rows': len(skill_demand_monthly), 'columns': skill_demand_monthly.shape[1], 'grain': 'month-skill aggregate'},
])

display(file_inventory)
print('Core jobs columns')
display(jobs[['job_title', 'seniority', 'country', 'salary_mid_usd', 'remote_type', 'ai_focus_area', 'required_skills']].head())

print('Top job titles')
display(jobs['job_title'].value_counts().head(10).rename_axis('job_title').reset_index(name='postings'))

## 5. Data Quality Check <a id='quality'></a>

A salary model is only as trustworthy as the columns feeding it. We check missingness across the `jobs` columns we plan to use and verify the target is clean. The bar chart highlights any field where imputation would be needed before modelling.

In [ ]:
model_cols = [
    'salary_mid_usd', 'job_title', 'role_family', 'seniority', 'remote_type',
    'country', 'region', 'industry', 'company_size', 'funding_stage',
    'experience_min_years', 'skills_count', 'visa_sponsorship', 'equity_offered',
]

missing = jobs[model_cols].isna().mean().mul(100).round(2).sort_values(ascending=False)
print('Percent missing in modelling columns:')
display(missing.rename('pct_missing').reset_index().rename(columns={'index': 'column'}))

print(f"Target salary_mid_usd -> nulls: {jobs['salary_mid_usd'].isna().sum()}, "
      f"min: {jobs['salary_mid_usd'].min():,.0f}, max: {jobs['salary_mid_usd'].max():,.0f}")

fig, ax = plt.subplots(figsize=(9, 5))
missing.plot(kind='barh', ax=ax, color='#c0504d')
ax.set_title('Missingness across modelling columns (%)')
ax.set_xlabel('Percent missing')
ax.set_ylabel('')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Role and Salary Landscape <a id='roles'></a>

We first orient ourselves on *volume* (which roles are posted most) and *pay* (median salary by role and by country). This frames every downstream comparison.

In [ ]:
role_counts = jobs['job_title'].value_counts().head(10).sort_values()
role_salary = jobs.groupby('job_title', as_index=False)['salary_mid_usd'].median().sort_values('salary_mid_usd', ascending=False).head(10)
country_salary = jobs.groupby('country', as_index=False)['salary_mid_usd'].median().sort_values('salary_mid_usd', ascending=False).head(10)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
role_counts.plot(kind='barh', ax=axes[0], title='Top job titles by posting volume')
axes[0].set_xlabel('Postings')
axes[0].set_ylabel('')

sns.barplot(data=role_salary, x='salary_mid_usd', y='job_title', ax=axes[1])
axes[1].set_title('Median salary by role')
axes[1].set_xlabel('Median salary_mid_usd')
axes[1].set_ylabel('')

sns.barplot(data=country_salary, x='salary_mid_usd', y='country', ax=axes[2])
axes[2].set_title('Median salary by country')
axes[2].set_xlabel('Median salary_mid_usd')
axes[2].set_ylabel('')

plt.tight_layout()
plt.show()

salary_snapshot = jobs.groupby(['job_title', 'seniority'], as_index=False)['salary_mid_usd'].median()
display(salary_snapshot.sort_values('salary_mid_usd', ascending=False).head(12))

## 7. Salary Distribution (EDA) <a id='dist'></a>

The shape of the salary distribution determines how we model it. Compensation data is almost always right-skewed, which matters: a linear model on the raw target will chase the long tail. We plot the raw distribution alongside a log-transformed version to assess whether a log target is justified.

In [ ]:
salary = jobs['salary_mid_usd'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(salary, bins=50, kde=True, ax=axes[0], color='#4f81bd')
axes[0].axvline(salary.median(), color='black', linestyle='--', label=f'median ${salary.median():,.0f}')
axes[0].set_title('Raw salary_mid_usd distribution')
axes[0].set_xlabel('salary_mid_usd')
axes[0].legend()

sns.histplot(np.log1p(salary), bins=50, kde=True, ax=axes[1], color='#9bbb59')
axes[1].set_title('log1p(salary_mid_usd) distribution')
axes[1].set_xlabel('log1p(salary_mid_usd)')

plt.tight_layout()
plt.show()

print(f'Raw skew      : {salary.skew():.3f}')
print(f'log1p skew    : {np.log1p(salary).skew():.3f}')
print(f'Median / mean : ${salary.median():,.0f} / ${salary.mean():,.0f}')

## 8. Salary by Seniority (EDA) <a id='seniority'></a>

Seniority is the single strongest lever on pay in this market. A boxplot per level shows both the central tendency and the spread &mdash; the widening interquartile range at staff/principal levels is itself a finding about how compensation negotiation opens up at the top.

In [ ]:
seniority_order = ['entry', 'mid', 'senior', 'staff', 'principal']
present_levels = [s for s in seniority_order if s in jobs['seniority'].unique()]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.boxplot(data=jobs, x='seniority', y='salary_mid_usd', order=present_levels, ax=axes[0])
axes[0].set_title('Salary by seniority')
axes[0].set_xlabel('')
axes[0].set_ylabel('salary_mid_usd')

sns.boxplot(data=jobs, x='remote_type', y='salary_mid_usd', ax=axes[1])
axes[1].set_title('Salary by remote arrangement')
axes[1].set_xlabel('')
axes[1].set_ylabel('salary_mid_usd')

plt.tight_layout()
plt.show()

med_by_level = jobs.groupby('seniority')['salary_mid_usd'].median().reindex(present_levels)
print('Median salary by seniority:')
display(med_by_level.rename('median_salary').reset_index())

## 9. Skill Demand and Trend Shifts <a id='skills'></a>

Using the monthly aggregate table we track how the *share* of tracked roles shifts over 2024-2026 and which skills lead demand in the most recent month. This is the time-series view of the market.

In [ ]:
jobs['posted_month'] = pd.to_datetime(jobs['posted_date']).dt.to_period('M').astype(str)
role_mix = (
    jobs[jobs['job_title'].isin(['AI Engineer', 'LLM Engineer', 'ML Engineer', 'Data Scientist'])]
    .groupby(['posted_month', 'job_title'])['job_id']
    .count()
    .reset_index(name='postings')
)
role_pivot = role_mix.pivot(index='posted_month', columns='job_title', values='postings').fillna(0)
role_share = role_pivot.div(role_pivot.sum(axis=1), axis=0)

latest_skills = (
    skill_demand_monthly[skill_demand_monthly['year_month'] == skill_demand_monthly['year_month'].max()]
    .sort_values('job_count', ascending=False)
    .head(12)
)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
role_share[['AI Engineer', 'LLM Engineer', 'ML Engineer', 'Data Scientist']].plot(ax=axes[0])
axes[0].set_title('Role share shift over time')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Share of tracked roles')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(data=latest_skills, x='job_count', y='skill', ax=axes[1])
axes[1].set_title(f"Top skills in {skill_demand_monthly['year_month'].max()}")
axes[1].set_xlabel('Job count')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

display(latest_skills[['skill', 'skill_category', 'job_count', 'median_salary_mid_usd', 'remote_share', 'share_of_postings']])

## 10. Skill Ranking via the Edge Table <a id='ranking'></a>

The `job_skills` table is a bipartite job-to-skill graph with ~318K edges. Counting edges per skill is the cleanest demand ranking we have because it reflects *every* listing's stated requirements, not a parsed text field. We also break demand down by skill category and flag GenAI-specific skills.

In [ ]:
n_jobs = job_skills['job_id'].nunique()
skill_rank = (
    job_skills.groupby('skill')
    .agg(edge_count=('job_id', 'size'),
         is_genai=('is_genai_skill', 'max'),
         core_share=('importance', lambda s: (s == 'core').mean()))
    .reset_index()
)
skill_rank['pct_of_jobs'] = (skill_rank['edge_count'] / n_jobs * 100).round(1)
skill_rank = skill_rank.sort_values('edge_count', ascending=False)
top_skills = skill_rank.head(15)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
colors = ['#e07b39' if g else '#4f81bd' for g in top_skills['is_genai']]
axes[0].barh(top_skills['skill'][::-1], top_skills['edge_count'][::-1], color=colors[::-1])
axes[0].set_title('Top 15 skills by edge count (orange = GenAI skill)')
axes[0].set_xlabel('Job-skill edges')

cat_demand = job_skills['skill_category'].value_counts().sort_values()
cat_demand.plot(kind='barh', ax=axes[1], color='#8064a2')
axes[1].set_title('Edge volume by skill category')
axes[1].set_xlabel('Job-skill edges')

plt.tight_layout()
plt.show()

print(f'Total edges: {len(job_skills):,} across {n_jobs:,} jobs and {job_skills["skill"].nunique()} distinct skills')
display(top_skills[['skill', 'edge_count', 'pct_of_jobs', 'core_share', 'is_genai']])

## 11. Skill Pay Premiums <a id='premium'></a>

Demand and pay are not the same thing. By joining the edge table back to `jobs` we can compute the median salary of listings that require each skill and compare it to the overall market median. The *premium* (skill median minus market median) tells us which skills command a wage above baseline &mdash; the practical question a job-seeker actually cares about.

In [ ]:
market_median = jobs['salary_mid_usd'].median()
edge_sal = job_skills.merge(jobs[['job_id', 'salary_mid_usd']], on='job_id', how='left')

skill_pay = (
    edge_sal.groupby('skill')
    .agg(median_salary=('salary_mid_usd', 'median'), demand=('job_id', 'size'))
    .reset_index()
)
# Focus on reasonably common skills so a handful of postings cannot dominate.
skill_pay = skill_pay[skill_pay['demand'] >= 500].copy()
skill_pay['premium'] = (skill_pay['median_salary'] - market_median).round(0)
top_premium = skill_pay.sort_values('premium', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=top_premium, x='premium', y='skill', ax=ax, palette='viridis')
ax.axvline(0, color='black', linewidth=1)
ax.set_title(f'Salary premium vs market median (${market_median:,.0f})')
ax.set_xlabel('Median salary above market (USD)')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

display(top_premium[['skill', 'median_salary', 'premium', 'demand']])

## 12. Benchmarks and Remote Mix <a id='benchmarks'></a>

The pre-aggregated `salary_benchmarks` table gives percentile bands per role/country/seniority slice, and the remote breakdown shows how flexible this market is. These are ready-made views for dashboards and offer-negotiation context.

In [ ]:
remote_mix = jobs['remote_type'].value_counts(normalize=True).rename('share').mul(100).round(1)
country_remote = (
    jobs.assign(is_remote=(jobs['remote_type'] == 'remote').astype(int))
    .groupby('country', as_index=False)
    .agg(remote_share=('is_remote', 'mean'), median_salary=('salary_mid_usd', 'median'), postings=('job_id', 'count'))
    .sort_values(['postings', 'median_salary'], ascending=[False, False])
)

print('Remote mix (%)')
display(remote_mix.reset_index().rename(columns={'index': 'remote_type'}))

print('Top benchmark slices by median salary')
display(salary_benchmarks.sort_values('salary_median_usd', ascending=False).head(12))

print('Country remote-share view')
display(country_remote.head(12))

## 13. Method: Salary Regression <a id='method'></a>

We now build a salary model with a transparent, defensible feature set:

- **Categorical:** `seniority`, `role_family`, `remote_type`, `region`, `country`, `industry`, `company_size`, `funding_stage` &mdash; one-hot encoded.
- **Numeric:** `experience_min_years`, `skills_count`, `visa_sponsorship`, `equity_offered`.

Because the target is right-skewed (see Section 7) we model **`log1p(salary_mid_usd)`** and back-transform predictions for reporting. We compare a `Ridge` linear baseline against a `HistGradientBoostingRegressor`, both inside a scikit-learn `Pipeline` so preprocessing is fit only on training folds. Every estimator and split uses `random_state=SEED`.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_features = ['seniority', 'role_family', 'remote_type', 'region', 'country',
                'industry', 'company_size', 'funding_stage']
num_features = ['experience_min_years', 'skills_count', 'visa_sponsorship', 'equity_offered']

model_df = jobs.dropna(subset=['salary_mid_usd'] + cat_features + num_features).copy()
X = model_df[cat_features + num_features]
y = np.log1p(model_df['salary_mid_usd'])  # model the log target

# sparse_output=False keeps the matrix dense so HistGradientBoostingRegressor accepts it.
preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
    ('num', StandardScaler(), num_features),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

ridge = Pipeline([('prep', preprocess), ('model', Ridge(alpha=1.0, random_state=SEED))])
hgb = Pipeline([('prep', preprocess),
                ('model', HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08,
                                                         max_depth=None, random_state=SEED))])

print(f'Modelling rows: {len(model_df):,}  |  features: {len(cat_features)} categorical + {len(num_features)} numeric')
print(f'Train / test split: {len(X_train):,} / {len(X_test):,} (random_state=SEED)')

## 14. Evaluation <a id='evaluation'></a>

We score both models with 5-fold cross-validation on the training set, then confirm the chosen model on the held-out test set. Metrics are reported in **dollar space** (RMSE, MAE) after back-transforming `log1p` predictions, plus R<sup>2</sup> in log space. Reporting error in dollars keeps the result interpretable for anyone reading a salary range.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

def evaluate(name, pipe):
    # Out-of-fold predictions in log space, back-transformed to dollars for RMSE/MAE.
    oof_log = cross_val_predict(pipe, X_train, y_train, cv=kf)
    oof_usd = np.expm1(oof_log)
    true_usd = np.expm1(y_train)
    rmse = np.sqrt(mean_squared_error(true_usd, oof_usd))
    mae = mean_absolute_error(true_usd, oof_usd)
    r2 = r2_score(y_train, oof_log)
    return {'model': name, 'cv_rmse_usd': round(rmse), 'cv_mae_usd': round(mae), 'cv_r2_log': round(r2, 3)}

results = pd.DataFrame([evaluate('Ridge', ridge), evaluate('HistGBR', hgb)])
print('5-fold cross-validation (training set):')
display(results)

best_name = results.sort_values('cv_rmse_usd').iloc[0]['model']
best_pipe = ridge if best_name == 'Ridge' else hgb
best_pipe.fit(X_train, y_train)
test_pred_usd = np.expm1(best_pipe.predict(X_test))
test_true_usd = np.expm1(y_test)

print(f'\nHeld-out test ({best_name}):')
print(f'  RMSE : ${np.sqrt(mean_squared_error(test_true_usd, test_pred_usd)):,.0f}')
print(f'  MAE  : ${mean_absolute_error(test_true_usd, test_pred_usd):,.0f}')
print(f'  R2   : {r2_score(y_test, best_pipe.predict(X_test)):.3f} (log space)')

## 15. Diagnostics: Residuals & Feature Importance <a id='diagnostics'></a>

Two diagnostics close the modelling loop. The **residual plot** (predicted vs actual, in dollars) shows whether errors are unbiased across the salary range. The **permutation importance** chart ranks which feature groups actually drive predictions &mdash; a model-agnostic check that the model is learning the economically sensible signals (seniority, geography) rather than noise.

In [ ]:
from sklearn.inspection import permutation_importance

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

# Residual / calibration scatter on a reproducible sample of the test set.
sample_idx = rng.choice(len(test_true_usd), size=min(3000, len(test_true_usd)), replace=False)
axes[0].scatter(test_true_usd.to_numpy()[sample_idx], test_pred_usd[sample_idx],
                alpha=0.2, s=12, color='#4f81bd')
lims = [test_true_usd.min(), test_true_usd.max()]
axes[0].plot(lims, lims, 'k--', linewidth=1.5, label='perfect prediction')
axes[0].set_title(f'Predicted vs actual salary ({best_name})')
axes[0].set_xlabel('Actual salary_mid_usd')
axes[0].set_ylabel('Predicted salary_mid_usd')
axes[0].legend()

# Permutation importance grouped to the original feature names.
perm = permutation_importance(best_pipe, X_test, y_test, n_repeats=5,
                              random_state=SEED, scoring='r2')
imp = (pd.Series(perm.importances_mean, index=X_test.columns)
       .sort_values().tail(12))
imp.plot(kind='barh', ax=axes[1], color='#9bbb59')
axes[1].set_title('Permutation importance (drop in R2)')
axes[1].set_xlabel('Mean importance')

plt.tight_layout()
plt.show()

## 16. Insights <a id='insights'></a>

Pulling the EDA and modelling together, several interpretations of the AI/data labour market stand out:

1. **Seniority dominates pay, and permutation importance confirms it.** The boxplots show a clean monotonic climb from entry to principal, and the model's importance ranking puts `seniority` at the top. *Therefore* a salary estimate that ignores level is almost meaningless; level is the first question to pin down.

2. **Demand and pay diverge for some skills.** Python and SQL top the edge-count ranking, but the skill *premium* chart shows that ubiquitous skills sit near the market median &mdash; they are table stakes, not differentiators. The observation that the highest premiums attach to scarcer, specialised skills is the actionable finding for upskilling: rarity, not popularity, pays.

3. **GenAI/LLM roles are reshaping the role mix.** The role-share trend shows AI Engineer and LLM Engineer gaining share across 2024-2026. *Because* the dataset spans the post-ChatGPT hiring wave, this is a genuine market signal rather than sampling noise &mdash; though it is still a hypothesis worth confirming against external job-board data.

4. **Geography is a first-order salary lever.** Median salary by country spans a wide band, and `country`/`region` rank highly in permutation importance. The trade-off for employers is real: remote-friendly roles widen the candidate pool but compress the geographic salary arbitrage.

5. **A log target was the right call.** The raw distribution is strongly right-skewed; modelling `log1p(salary)` and reporting error in dollars gave well-calibrated residuals across the range (see the predicted-vs-actual plot), an interpretation backed by the much lower skew of the transformed target.

**Caveats and limitations.** This is a synthetic/benchmark-style dataset, so absolute salary levels should be read as *relative* signals, not authoritative market rates. The model uses only structured fields &mdash; the free-text `description` and the rich `required_skills` list are left on the table, which is a clear limitation. Finally, the edge-count ranking treats every required skill equally; weighting by `importance` (core vs nice-to-have) would refine it.

## Key Findings (dataset utility)

- The dataset is strong for **salary modeling** because compensation moves cleanly with seniority, geography, company size, and role family &mdash; exactly the levers the regression rewards.
- `job_skills.csv` makes the package immediately useful for **graph ML**, retrieval, and skill recommendation work instead of forcing skill parsing from raw text.
- The monthly aggregate tables make it easy to study the 2024-2026 shift toward **AI Engineer** and **LLM Engineer** roles without rebuilding time-based features from scratch.
- `salary_benchmarks.csv` is already presentation-ready for dashboards and compensation benchmarking workflows.

## 17. Conclusion & Next Steps <a id='conclusion'></a>

**Summary.** Across five linked tables we profiled the AI/data jobs market, established that seniority and geography are the dominant pay levers, separated *in-demand* skills from *premium-paying* ones via the 318K-edge graph, tracked the 2024-2026 rise of GenAI roles, and fit a cross-validated salary regressor on a log target with interpretable dollar-space error and sensible permutation-importance. The headline takeaway: in this market, **level and location set the band, and scarce specialised skills set the premium.**

**Next steps and recommended improvements.**

- **Enrich the feature set.** Bring the `required_skills` list and free-text `description` into the model via multi-hot skill flags or TF-IDF embeddings; this is the most promising path to lower RMSE and is the obvious future work given the limitations above.
- **Add a time-aware split.** Train on 2024-2025 and validate on 2026 listings to recommend whether the model generalises to the newest GenAI-heavy postings rather than only to a random hold-out.
- **Weight the skill ranking.** Recompute demand and premiums using the `importance` field so core requirements count more than nice-to-haves, improving the skill recommendations.
- **Join company signals.** Merge `companies.csv` (ai_maturity_score, glassdoor_like_rating, hiring_velocity_index) to test whether employer quality explains residual salary variance.
- **Productionise as a benchmark API.** Wrap the trained pipeline behind a simple function that returns a percentile salary band for a given role/level/country &mdash; the natural deliverable for a compensation tool.

**Final thoughts.** The notebook is intentionally reproducible end to end (single `SEED`, `random_state` everywhere): re-running it should regenerate every chart, metric, and ranking, giving a trustworthy foundation for the deeper modelling described above.